In [0]:
# ==========================================
# 01. BRONZE SAMPLE & SETUP
# ==========================================
from pyspark.sql.functions import col

# 1. Widgets for dynamic configuration
dbutils.widgets.text("env", "dev", "1. Environment")
dbutils.widgets.text("catalog", "workspace", "2. Catalog")
dbutils.widgets.text("schema_name", "default", "3. Schema Name")
dbutils.widgets.text("secret_scope", "lab-secrets-scope", "4. Secret Scope")

env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema_name")
scope_name = dbutils.widgets.get("secret_scope")

# 2. Ensure Schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

# 3. Define Bronze source table name
bronze_table = f"{catalog}.{schema_name}.wiki_bronze_data"

# 4. Prepare mock Bronze dataset (with duplicates & null keys)
mock_bronze_data = [
    (101, "Main Page", "UserA", False, "initial edit", "enwiki", 1700000000),
    (101, "Main Page", "UserA", False, "initial edit duplicate", "enwiki", 1700000000),
    (102, "Apache Spark", "UserB", True, "bot cleanup", "ukwiki", 1700000100),
    (103, "Delta Lake", "UserC", False, "added architecture section", "enwiki", 1700000200),
    (None, "Invalid Page", "Anon", False, "corrupted record", "enwiki", 1700000300)
]

columns = ["id", "title", "user", "bot", "comment", "wiki", "timestamp"]
df_bronze_init = spark.createDataFrame(mock_bronze_data, columns)

# 5. Save to Bronze Delta table
df_bronze_init.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"✅ Step 1 Complete! Raw Bronze table created: {bronze_table}")
display(spark.table(bronze_table))